# 04 — Score External Outputs with Biomni-Eval1

This notebook lets you score agent outputs that were **collected outside bio-agents-playground** — from another tool, a third-party API, a manual run, or any external source — using the exact same `BiomniEval1` scorer as `run_eval1.py`.

### Workflow

```
External outputs (dict / JSONL / CSV)
          ↓
  StandaloneEval1Scorer
          ↓
  score, passed, metrics
          ↓
  pandas analysis → results.jsonl
```

### Input format

Each record needs exactly three required fields:

| Field | Type | Description |
|---|---|---|
| `task_name` | str | One of the 10 Biomni-Eval1 task types |
| `task_instance_id` | int | Row ID within that task type in `biomni/Eval1` |
| `output` | str | The agent's free-text response |

Any extra fields (`framework`, `model`, `run_id`, `cost`, …) are passed through untouched.

### Scoring

Binary exact match: **0.0** (incorrect) or **1.0** (correct). Task-specific rules:
- Gene symbol tasks → case-insensitive exact match
- Multiple-choice tasks (`lab_bench_*`, `crispr_delivery`) → single-letter exact match
- `rare_disease_diagnosis` → JSON parse + OMIM ID comparison
- `patient_gene_detection` → JSON parse + gene list intersection

### Prerequisites
```bash
uv sync --extra biomni --extra dev
```

In [ ]:
import sys
from pathlib import Path

import pandas as pd

repo_root = Path(".").resolve().parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

print(f"Repo root: {repo_root}")

## 2. Prepare your outputs

You can supply outputs as:
- A plain Python list of dicts (this section)
- A JSONL file — see Section 4
- A CSV file — see Section 4

Edit the list below with your own records. The `task_instance_id` values must exist in the `biomni/Eval1` dataset for the given `task_name`.

To look up valid `(task_name, task_instance_id)` pairs, load the dataset:
```python
from datasets import load_dataset
ds = load_dataset("biomni/Eval1", split="test")
df = ds.to_pandas()
df[df["task_name"] == "gwas_causal_gene_opentargets"][["task_instance_id", "prompt"]].head()
```

In [ ]:
# Edit these records with your own outputs.
# The three required fields are task_name, task_instance_id, and output.
# Extra fields (framework, model, run_id, ...) are optional but preserved in results.

records = [
    {
        "task_name": "gwas_causal_gene_opentargets",
        "task_instance_id": 767,
        "output": "Based on the evidence, the most likely causal gene is HNF1A.",
        "framework": "my_external_tool",
        "model": "gpt-4o",
    },
    {
        "task_name": "lab_bench_seqqa",
        "task_instance_id": 12,
        "output": "The answer is B.",
        "framework": "my_external_tool",
        "model": "gpt-4o",
    },
    {
        "task_name": "crispr_delivery",
        "task_instance_id": 5,
        "output": "For this application, delivery method C is most appropriate.",
        "framework": "my_external_tool",
        "model": "gpt-4o",
    },
]

pd.DataFrame(records)

## 3. Score with `StandaloneEval1Scorer`

In [ ]:
from bio_agents.evaluation.standalone_scorer import StandaloneEval1Scorer

scorer = StandaloneEval1Scorer()

results = scorer.score_batch(records)
df = pd.DataFrame(results)

df[["task_name", "task_instance_id", "framework", "model", "score", "passed"]]

In [ ]:
# Overall and per-task summary
stats = scorer.summary(results)

print(
    f"Overall: {stats['n_passed']}/{stats['n_total']} passed  "
    f"| Avg score: {stats['avg_score']:.3f}  "
    f"| Pass rate: {stats['pass_rate']:.0%}"
)
print()

per_task_df = pd.DataFrame(stats["per_task"]).T
per_task_df

## 4. Load from a file (JSONL or CSV)

If your outputs are already saved to disk, use the file loader methods directly.
Both formats require the same three fields; extra columns are preserved.

In [ ]:
# --- JSONL loader ---
# Expects one JSON object per line:
#   {"task_name": "...", "task_instance_id": 1, "output": "...", ...}

JSONL_PATH = repo_root / "results" / "external_outputs.jsonl"  # edit this path

if JSONL_PATH.exists():
    results_from_jsonl = scorer.score_from_jsonl(JSONL_PATH)
    pd.DataFrame(results_from_jsonl)[
        ["task_name", "task_instance_id", "score", "passed"]
    ]
else:
    print(f"File not found: {JSONL_PATH}\nEdit the path above or skip this cell.")

In [ ]:
# --- CSV loader ---
# Expects a header row with at least: task_name, task_instance_id, output

CSV_PATH = repo_root / "results" / "external_outputs.csv"  # edit this path

if CSV_PATH.exists():
    results_from_csv = scorer.score_from_csv(CSV_PATH)
    pd.DataFrame(results_from_csv)[["task_name", "task_instance_id", "score", "passed"]]
else:
    print(f"File not found: {CSV_PATH}\nEdit the path above or skip this cell.")

## 5. Score a single output

`score_one()` is useful for quick interactive checks or when building a scoring loop manually.

In [ ]:
single = scorer.score_one(
    task_name="gwas_causal_gene_opentargets",
    task_instance_id=767,
    output="The causal gene in this locus is HNF1A.",
    model="my-model",  # optional extra field
)

print(f"Score  : {single['score']}")
print(f"Passed : {single['passed']}")
print(f"Metrics: {single['metrics']}")

## 6. Analyse results

In [ ]:
# Score breakdown per task type
df.groupby("task_name")[["score", "passed"]].agg(
    avg_score=("score", "mean"),
    n_passed=("passed", "sum"),
    n_total=("passed", "count"),
).assign(pass_rate=lambda x: x["n_passed"] / x["n_total"]).sort_values(
    "avg_score", ascending=False
)

In [ ]:
# Score breakdown per model (useful when comparing multiple models)
if "model" in df.columns:
    df.groupby("model")[["score"]].agg(
        avg_score=("score", "mean"),
        n_total=("score", "count"),
    )
else:
    print("No 'model' column in results — add it as an extra field to your records.")

In [ ]:
# Inspect a single result in detail
r = results[0]
print(f"task_name        : {r['task_name']}")
print(f"task_instance_id : {r['task_instance_id']}")
print(f"score            : {r['score']}")
print(f"passed           : {r['passed']}")
print(f"metrics          : {r['metrics']}")
print("\n--- Output (first 500 chars) ---")
print(str(r.get("output", ""))[:500])

## 7. Save and reload results

Results are written as JSONL — the same format as `BenchmarkRunner` and `Eval1Runner` — so you can load and compare runs from different sources in one DataFrame.

In [ ]:
OUTPUT_DIR = repo_root / "results" / "external_scored_run"  # edit as needed

out_file = scorer.save(results, output_dir=OUTPUT_DIR)
print(f"Saved {len(results)} result(s) to: {out_file}")

In [ ]:
# Reload from JSONL — handy for comparing across sessions or merging runs
df_saved = pd.read_json(out_file, lines=True)
print(f"Rows in file: {len(df_saved)}")
df_saved[["task_name", "task_instance_id", "score", "passed"]]

## 8. CLI equivalent

The same scorer is available as a CLI script — useful for batch scoring outside a notebook:

```bash
# Score a JSONL file
uv run python pipelines/score_outputs.py \
    --input path/to/external_outputs.jsonl \
    --output results/my_scored_run

# Score a CSV file
uv run python pipelines/score_outputs.py \
    --input path/to/external_outputs.csv \
    --output results/my_scored_run

# Dry-run: preview input records without scoring
uv run python pipelines/score_outputs.py \
    --input path/to/external_outputs.jsonl \
    --dry-run
```

### Input file format (JSONL)

```jsonl
{"task_name": "gwas_causal_gene_opentargets", "task_instance_id": 767, "output": "The causal gene is HNF1A.", "model": "gpt-4o"}
{"task_name": "lab_bench_seqqa", "task_instance_id": 12, "output": "The answer is B.", "model": "gpt-4o"}
```

### Merge with Eval1Runner results for cross-framework comparison

```python
import pandas as pd

df_internal = pd.read_json("results/biomni_eval1_smoke/results.jsonl", lines=True)
df_external = pd.read_json("results/external_scored_run/results.jsonl", lines=True)

df_all = pd.concat([df_internal, df_external], ignore_index=True)
df_all.groupby(["framework", "model"])["score"].mean()
```